In [49]:
import jax
import jax.numpy as jnp
from jax import lax


@jax.jit
def solve_eta(a_s, t_final, kernel_prefactor):
    a_s = jnp.asarray(a_s, dtype=jnp.complex64)

    num_points = a_s.shape[0]
    num_steps = num_points - 1
    dt = t_final / num_steps

    eta_0 = -4.0 * jnp.pi * a_s[0]
    eta = jnp.zeros_like(a_s, dtype=jnp.complex64).at[0].set(eta_0)

    l1_prefactor = 2.0 * kernel_prefactor / jnp.sqrt(dt)
    j = jnp.arange(num_steps)

    def time_step(history, k):
        diffs = history[1:] - history[:-1]

        # Only j = 0, ..., k - 2 is known. The j = k - 1 term contains eta[k]
        # and is moved into the denominator below.
        valid = j < (k - 1)
        m = k - j
        safe_m = jnp.maximum(m, 1)
        weights = jnp.where(
            valid,
            jnp.sqrt(safe_m) - jnp.sqrt(safe_m - 1),
            0.0,
        )

        known_history = jnp.sum(weights * diffs)
        numerator = -1.0 + l1_prefactor * (history[k - 1] - known_history)
        denominator = 1.0 / (4.0 * jnp.pi * a_s[k]) + l1_prefactor
        eta_k = numerator / denominator

        history = history.at[k].set(eta_k)
        return history, eta_k

    indices = jnp.arange(1, num_points)
    eta, _ = lax.scan(time_step, eta, indices)

    return eta


def solve_eta_reference(a_s, t_final, kernel_prefactor):
    a_s = jnp.asarray(a_s, dtype=jnp.complex64)

    num_points = a_s.shape[0]
    num_steps = num_points - 1
    dt = t_final / num_steps
    l1_prefactor = 2.0 * kernel_prefactor / jnp.sqrt(dt)

    eta = jnp.zeros_like(a_s, dtype=jnp.complex64)
    eta = eta.at[0].set(-4.0 * jnp.pi * a_s[0])

    for k in range(1, num_points):
        known_history = 0.0 + 0.0j
        for jj in range(k - 1):
            weight = jnp.sqrt(k - jj) - jnp.sqrt(k - jj - 1)
            known_history = known_history + weight * (eta[jj + 1] - eta[jj])

        numerator = -1.0 + l1_prefactor * (eta[k - 1] - known_history)
        denominator = 1.0 / (4.0 * jnp.pi * a_s[k]) + l1_prefactor
        eta = eta.at[k].set(numerator / denominator)

    return eta


def assert_eta_case(name, a_s, t_final, kernel_prefactor):
    eta = solve_eta(a_s, t_final, kernel_prefactor)
    eta_ref = solve_eta_reference(a_s, t_final, kernel_prefactor)
    eta.block_until_ready()

    max_error = jnp.max(jnp.abs(eta - eta_ref))

    assert eta.shape == a_s.shape, f"{name}: wrong shape"
    assert eta.dtype == jnp.complex64, f"{name}: wrong dtype"
    assert bool(jnp.all(jnp.isfinite(eta)).item()), f"{name}: non-finite eta"
    assert bool(jnp.allclose(eta, eta_ref, rtol=2e-5, atol=2e-5).item()), (
        f"{name}: does not match reference, max error = {max_error}"
    )

    return eta


nondimensional_prefactor = -1.0 / (4.0 * jnp.pi ** (3 / 2) * jnp.sqrt(1j))
t_final = 1.0
num_steps = 10
time_grid = jnp.linspace(0.0, t_final, num_steps + 1)

a_s_1 = jnp.full(num_steps + 1, 0.20)
a_s_2 = jnp.linspace(0.10, 0.50, num_steps + 1)
a_s_3 = 0.20 + 0.05 * jnp.sin(2.0 * jnp.pi * time_grid)
a_s_4 = 0.20 + 0.03j * jnp.linspace(0.0, 1.0, num_steps + 1)
a_s_5 = jnp.linspace(0.12 + 0.02j, 0.42 + 0.08j, num_steps + 1)

test_cases = {
    "constant real": a_s_1,
    "ramped real": a_s_2,
    "oscillating real": a_s_3,
    "constant real plus imaginary ramp": a_s_4,
    "complex ramp": a_s_5,
}

print("a_s_1 eta:", solve_eta(test_cases["constant real"], t_final, nondimensional_prefactor)
print("a_s_2 eta:", solve_eta(test_cases["ramped real"], t_final, nondimensional_prefactor)
print("a_s_1 eta:", solve_eta(test_cases["oscillating real"], t_final, nondimensional_prefactor)
print("a_s_1 eta:", solve_eta(test_cases["constant real"], t_final, nondimensional_prefactor)


All eta solver tests passed.
a_s_1 eta: [-2.5132742+0.0000000e+00j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j -2.5132742-1.0580138e-07j
 -2.5132742-1.0580138e-07j]
